In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the collected Bitcoin news dataset
news_path = Path("../data/news/raw/bitcoin_news_72h.csv")
news_df = pd.read_csv(news_path)

print("========== Dataset Overview ==========")
print("File:", news_path.resolve())
print("Total articles:", len(news_df))
print("Columns:", news_df.columns.tolist())

print("\n========== First 5 Articles ==========")
display(news_df.head())

========== Dataset Overview ==========
File: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\news\raw\bitcoin_news_72h.csv
Total articles: 129
Columns: ['title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'content_status']

========== First 5 Articles ==========


,title,source,published_at_utc,link,summary,content,content_status
0,Bitcoin ETFs lose $679 million over the week,Yahoo Finance Video,2026-10-12T17:00:00+00:00,https://finance.yahoo.com/video/bitcoin-etfs-l...,Scott Melker discusses how bitcoin ETFs lost $...,Scott Melker discusses how bitcoin ETFs lost $...,summary_and_content
1,The Whale Behind $29.75 Million in Wins Just W...,BeInCrypto,2026-10-13T07:51:27+00:00,https://finance.yahoo.com/markets/crypto/artic...,A Hyperliquid whale with $29.75 million in pro...,Leveraged bulls lost $688.56 million to liquid...,summary_and_content
2,"Cleveland Fed President Wants Small, Quick Rat...",BeInCrypto,2026-10-13T06:21:09+00:00,https://finance.yahoo.com/markets/crypto/artic...,Fed’s Hammack backs restrictive policy as Dece...,Cleveland Fed President Beth Hammack said that...,summary_and_content
3,Bulls Absorb 89% of $774 Million Crypto Liquid...,BeInCrypto,2026-10-13T01:59:15+00:00,https://finance.yahoo.com/markets/crypto/artic...,"Crypto liquidations hit $774 million, with lon...",Leveraged bulls took the brunt of a $774.14 mi...,summary_and_content
4,When Will Bitcoin Price Reclaim Its All-Time H...,BeInCrypto,2026-10-12T23:13:49+00:00,https://finance.yahoo.com/markets/crypto/artic...,Cycle data shows how fast Bitcoin could reclai...,Bitcoin price currently trades 35% below its r...,summary_and_content


In [3]:
# Check for duplicate records and missing values

print("========== 1. Duplicate Check ==========")
print("Duplicate links:", news_df["link"].duplicated().sum())
print("Duplicate titles:", news_df["title"].duplicated().sum())

print("\n========== 2. Missing Values ==========")
print(news_df[
    ["title", "source", "published_at_utc",
     "link", "summary", "content", "content_status"]
].isna().sum())

print("\n========== 3. Empty Text Check ==========")
print("Empty titles:",
      news_df["title"].fillna("").str.strip().eq("").sum())
print("Empty links:",
      news_df["link"].fillna("").str.strip().eq("").sum())
print("Empty content:",
      news_df["content"].fillna("").str.strip().eq("").sum())

print("\n========== 4. Content Status ==========")
print(news_df["content_status"].value_counts(dropna=False))

print("\n========== 5. Content Length ==========")
content_length = news_df["content"].fillna("").str.len()
print("Minimum content length:", content_length.min())
print("Content under 500 characters:", (content_length < 500).sum())

print("\n========== 6. Summary Check ==========")
summary = news_df["summary"].fillna("").str.strip()
content = news_df["content"].fillna("").str.strip()

print("Summary is None/empty:", summary.eq("").sum())
print("Summary identical to content:", (summary == content).sum())

========== 1. Duplicate Check ==========
Duplicate links: 0
Duplicate titles: 0

========== 2. Missing Values ==========
title               0
source              0
published_at_utc    0
link                0
summary             8
content             0
content_status      0
dtype: int64

========== 3. Empty Text Check ==========
Empty titles: 0
Empty links: 0
Empty content: 0

========== 4. Content Status ==========
content_status
summary_and_content    121
content_only             8
Name: count, dtype: int64

========== 5. Content Length ==========
Minimum content length: 100
Content under 500 characters: 8

========== 6. Summary Check ==========
Summary is None/empty: 8
Summary identical to content: 0


In [4]:
# Check publication timestamps against the latest 72-hour window

from datetime import datetime, timezone

check_time = datetime.now(timezone.utc)
cutoff_time = pd.Timestamp(check_time) - pd.Timedelta(hours=72)

news_df["published_at_utc"] = pd.to_datetime(
    news_df["published_at_utc"],
    utc=True,
    errors="coerce"
)

invalid_timestamps = news_df["published_at_utc"].isna()

outside_window = (
    (news_df["published_at_utc"] < cutoff_time) |
    (news_df["published_at_utc"] > pd.Timestamp(check_time))
) & ~invalid_timestamps

print("========== Publication Time Check ==========")
print("Check time (UTC):", check_time.isoformat())
print("72-hour cutoff (UTC):", cutoff_time.isoformat())
print("Invalid timestamps:", invalid_timestamps.sum())
print("Articles outside 72-hour window:", outside_window.sum())

valid_times = news_df.loc[
    ~invalid_timestamps, "published_at_utc"
]

if not valid_times.empty:
    print("Newest article (UTC):", valid_times.max())
    print("Oldest article (UTC):", valid_times.min())

if outside_window.any():
    print("\nArticles outside the 72-hour window:")
    display(
        news_df.loc[
            outside_window,
            ["title", "published_at_utc", "link"]
        ]
    )

# Filter articles within the latest 72 hours
recent_news_df = news_df.loc[
    news_df["published_at_utc"].between(
        cutoff_time,
        pd.Timestamp(check_time),
        inclusive="both"
    )
].copy()

print("Articles within the latest 72 hours:", len(recent_news_df))

========== Publication Time Check ==========
Check time (UTC): 2026-10-13T08:13:41.235360+00:00
72-hour cutoff (UTC): 2026-10-10T08:13:41.235360+00:00
Invalid timestamps: 0
Articles outside 72-hour window: 0
Newest article (UTC): 2026-10-13 07:51:27+00:00
Oldest article (UTC): 2026-10-10 09:48:00+00:00
Articles within the latest 72 hours: 129


In [5]:
# Identify articles that may be unrelated to Bitcoin or cryptocurrency

import re

crypto_keywords = [
    "bitcoin", "btc", "cryptocurrency", "crypto",
    "blockchain", "digital asset", "digital assets",
    "bitcoin etf", "crypto etf"
]

def check_crypto_relevance(row):
    text = f"{row['title']} {row['summary']} {row['content']}"
    text = text.lower()
    return any(
        re.search(r"\b" + re.escape(keyword) + r"\b", text)
        for keyword in crypto_keywords
    )

news_df["keyword_relevant"] = news_df.apply(
    check_crypto_relevance, axis=1
)

potentially_irrelevant = news_df[
    ~news_df["keyword_relevant"]
]

print("Total articles:", len(news_df))
print("Potentially relevant:", news_df["keyword_relevant"].sum())
print("Potentially irrelevant:", len(potentially_irrelevant))

display(
    potentially_irrelevant[
        ["title", "source", "summary"]
    ]
)

Total articles: 129
Potentially relevant: 124
Potentially irrelevant: 5


,title,source,summary
8,Sector Update: Financial Stocks Advance Late A...,MT Newswires,NaN
12,IREN Reportedly Joins Anthropic's Cloud Provid...,Investor's Business Daily,NaN
17,Sector Update: Financial Stocks Rise Monday Af...,MT Newswires,NaN
19,Anthropic Deal Report Gives IREN Stock Early B...,Barrons.com,NaN
39,"Exchange-Traded Funds Lower, Equity Futures Mi...",MT Newswires,NaN


In [6]:
# Inspect potentially irrelevant articles before making any decisions

review_columns = ["title", "source", "link", "content"]

for index, row in potentially_irrelevant.iterrows():
    print("\n" + "=" * 100)
    print(f"Index: {index}")
    print(f"Title: {row['title']}")
    print(f"Source: {row['source']}")
    print(f"Link: {row['link']}")
    print(f"Content length: {len(str(row['content']))}")
    print("\nFull content:")
    print(row["content"])


Index: 8
Title: Sector Update: Financial Stocks Advance Late Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-advance-afternoon-200011423.html
Content length: 100

Full content:
Financial stocks rose in late Monday afternoon trading, with the NYSE Financial Index adding 0.5% an

Index: 12
Title: IREN Reportedly Joins Anthropic's Cloud Providers; Shares Jump
Source: Investor's Business Daily
Link: https://finance.yahoo.com/m/04ab5ce8-c2c2-3b1c-952c-5ea1b7a9d653/iren-reportedly-joins.html
Content length: 123

Full content:
Anthropic reportedly signed a cloud services deal with IREN as it accelerates its data center buildouts. IREN stock surges.

Index: 17
Title: Sector Update: Financial Stocks Rise Monday Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-rise-monday-175415678.html
Content length: 100

Full content:
Financial stocks rose in Monday afternoon trading w

In [7]:
# Keyword-based filtering after the 72-hour date filter
crypto_keywords = [
    "bitcoin",
    "btc",
    "cryptocurrency",
    "cryptocurrencies",
    "crypto",
    "blockchain",
    "digital asset",
    "digital assets"
]

def is_crypto_related(row):
    fields = [
        row.get("title", ""),
        row.get("summary", ""),
        row.get("content", "")
    ]

    text = " ".join(
        "" if pd.isna(value) else str(value)
        for value in fields
    ).lower()

    return any(
        re.search(
            r"\b" + re.escape(keyword) + r"\b",
            text
        )
        for keyword in crypto_keywords
    )


# Keep the original datasets unchanged
keyword_mask = recent_news_df.apply(
    is_crypto_related, axis=1
)

keyword_filtered_df = recent_news_df.loc[
    keyword_mask
].copy()

excluded_news_df = recent_news_df.loc[
    ~keyword_mask
].copy()

print("========== Keyword Filtering Summary ==========")
print("Articles after date filtering:", len(recent_news_df))
print("Keyword-relevant articles:", len(keyword_filtered_df))
print("Articles excluded by keyword filter:", len(excluded_news_df))

print("\n========== Excluded Articles ==========")
display(
    excluded_news_df[
        ["title", "source", "published_at_utc"]
    ]
)


========== Keyword Filtering Summary ==========
Articles after date filtering: 129
Keyword-relevant articles: 124
Articles excluded by keyword filter: 5

========== Excluded Articles ==========


,title,source,published_at_utc
8,Sector Update: Financial Stocks Advance Late A...,MT Newswires,2026-10-12 20:00:11+00:00
12,IREN Reportedly Joins Anthropic's Cloud Provid...,Investor's Business Daily,2026-10-12 20:22:57+00:00
17,Sector Update: Financial Stocks Rise Monday Af...,MT Newswires,2026-10-12 17:54:15+00:00
19,Anthropic Deal Report Gives IREN Stock Early B...,Barrons.com,2026-10-12 18:58:00+00:00
39,"Exchange-Traded Funds Lower, Equity Futures Mi...",MT Newswires,2026-10-12 13:00:27+00:00


In [8]:
print("Date-filtered articles:", len(recent_news_df))
print("Keyword-filtered articles:", len(keyword_filtered_df))
print("Excluded articles:", len(excluded_news_df))

print("\nRetained article sample:")
display(
    keyword_filtered_df[
        ["title", "source", "published_at_utc"]
    ].head(5)
)

Date-filtered articles: 129
Keyword-filtered articles: 124
Excluded articles: 5

Retained article sample:


,title,source,published_at_utc
0,Bitcoin ETFs lose $679 million over the week,Yahoo Finance Video,2026-10-12 17:00:00+00:00
1,The Whale Behind $29.75 Million in Wins Just W...,BeInCrypto,2026-10-13 07:51:27+00:00
2,"Cleveland Fed President Wants Small, Quick Rat...",BeInCrypto,2026-10-13 06:21:09+00:00
3,Bulls Absorb 89% of $774 Million Crypto Liquid...,BeInCrypto,2026-10-13 01:59:15+00:00
4,When Will Bitcoin Price Reclaim Its All-Time H...,BeInCrypto,2026-10-12 23:13:49+00:00


## SBERT

In [9]:
# Load the SBERT model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully.")
print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())

INFO: No device provided, using cpu
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22

SBERT model loaded successfully.
Embedding dimension: 384


C:\Users\nyy32\AppData\Local\Temp\ipykernel_34420\4262035451.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())


In [10]:
# Define a neutral forecasting query
forecasting_query = """
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.
"""

# Prepare news text for SBERT
def build_article_text(row):
    text_parts = []

    for column in ["title", "summary", "content"]:
        value = row.get(column)

        if pd.notna(value) and str(value).strip():
            text_parts.append(
                f"{column.capitalize()}: {str(value).strip()}"
            )

    return "\n".join(text_parts)


news_texts = [
    build_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Query and Text Preparation ==========")
print("Forecasting query prepared:", bool(forecasting_query.strip()))
print("Candidate articles:", len(news_texts))
print("Empty article texts:", sum(not text.strip() for text in news_texts))

print("\n========== Forecasting Query ==========")
print(forecasting_query.strip())

print("\n========== First Article Text Preview ==========")
print(news_texts[0][:600] if news_texts else "No articles available.")

========== Query and Text Preparation ==========
Forecasting query prepared: True
Candidate articles: 124
Empty article texts: 0

========== Forecasting Query ==========
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.

========== First Article Text Preview ==========
Title: Bitcoin ETFs lose $679 million over the week
Summary: Scott Melker discusses how bitcoin ETFs lost $679 million over the past week, reversing the course of massive inflows.

"The Daily Wolf with Scott Melker" airs every day at 12:00 p.m. Tune in for your daily dose of all things crypto.

Make sure to also check out Yahoo Finance's new crypto hub to find the latest crypto-related news.
Content: Scott Melker discusses how bitcoin ETFs lost $679 million over the past week, reversing t

In [11]:
# 1. Encode the forecasting query
query_embedding = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Encode all candidate news articles
news_embeddings = sbert_model.encode(
    news_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Calculate cosine similarity
similarity_scores = util.cos_sim(
    query_embedding,
    news_embeddings
).cpu().numpy().flatten()

# 4. Create a separate results DataFrame
semantic_results_df = keyword_filtered_df.copy()

semantic_results_df["similarity_score"] = similarity_scores

# 5. Rank articles by semantic similarity
semantic_results_df = (
    semantic_results_df
    .sort_values(
        by="similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

semantic_results_df["similarity_rank"] = (
    range(1, len(semantic_results_df) + 1)
)

# 6. Display the retrieval summary
print("========== SBERT Retrieval Summary ==========")
print("Candidate articles:", len(news_texts))
print("Query embedding dimension:", query_embedding.shape[-1])
print("News embeddings shape:", tuple(news_embeddings.shape))
print("Similarity scores calculated:", len(similarity_scores))

print("\n========== Top 10 Retrieved Articles ==========")

display(
    semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ].head(10)
)

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.69it/s]

========== SBERT Retrieval Summary ==========
Candidate articles: 124
Query embedding dimension: 384
News embeddings shape: (124, 384)
Similarity scores calculated: 124

========== Top 10 Retrieved Articles ==========


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,0.626807
1,2,"Bitcoin and ethereum prices today, Monday, Oct...",Yahoo Personal Finance,2026-10-12 11:40:18+00:00,0.626133
2,3,Bitcoin slips to $82.6k as rise in oil prices ...,Investing.com,2026-10-12 07:08:56+00:00,0.605762
3,4,Bitcoin Ranges Near $83K as Traders Look to US...,decrypt,2026-10-12 11:41:50+00:00,0.584239
4,5,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,0.576363
5,6,"Why Bitcoin Can Fall Before a Fed Rate Hike, T...",MoneyLion,2026-10-12 17:00:27+00:00,0.565969
6,7,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,0.559304
7,8,"Crypto’s Crowded Longs Face a CPI Test, and a ...",CCN,2026-10-12 12:04:17+00:00,0.552340
8,9,Bitcoin Price Prediction: Golden Cross Signals...,FX Empire,2026-10-12 11:13:39+00:00,0.551598
9,10,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,0.544206


In [12]:
# Select the top 10 retrieved news articles

TOP_K = 10

retrieved_news_df = semantic_results_df.head(TOP_K).copy()

print("Retrieved news prepared successfully.")
print("Selected articles:", len(retrieved_news_df))
print(
    "Similarity score range:",
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

Retrieved news prepared successfully.
Selected articles: 10
Similarity score range: 0.5442 to 0.6268


In [13]:
print("========== Retrieved News Quality Check ==========")

print("Retrieved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)
print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

========== Retrieved News Quality Check ==========
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Scores sorted descending: True


In [14]:
# Review the relevance of the top 10 retrieved news articles

for _, row in retrieved_news_df.iterrows():
    print("\n" + "=" * 90)
    print("Rank:", row["similarity_rank"])
    print("Similarity score:", round(row["similarity_score"], 4))
    print("Title:", row["title"])
    print("Source:", row["source"])
    print("Published (UTC):", row["published_at_utc"])

    summary = row.get("summary")
    content = row.get("content")

    print("\nSummary:")
    print(summary if pd.notna(summary) else "None")

    print("\nContent preview:")
    print(str(content)[:600] if pd.notna(content) else "None")


Rank: 1
Similarity score: 0.6268
Title: Bitcoin Could Be Entering a New Bull Market Despite Latest Sell-Off
Source: BeInCrypto
Published (UTC): 2026-10-11 19:22:47+00:00

Summary:
BloFin Research says Bitcoin's new bull market remains intact despite the latest sell-off, as historical data suggests its boom-and-bust cycles are weakening.

Content preview:
Despite the market dip and liquidations over the past week, Bitcoin has seemingly entered a bull market, as the analysis of its 50-week moving average shows.

BloFin Research argues that Bitcoin's traditional four-year boom-and-bust cycle is gradually losing strength. Future rallies could deliver smaller gains, followed by shorter and less severe downturns.

The strongest evidence comes from Bitcoin's recovery above its 50-week moving average, a technical indicator that previously signaled bull-market transitions in 2015, 2019 and 2023.

Bitcoin reclaimed this level near $78,800 in September. 

Rank: 2
Similarity score: 0.6261
Title: 

In [15]:
# Remove identifiable promotional paragraphs from retrieval text only
ad_patterns = [
    r"Before you buy NVIDIA",
    r"stock picks are beating the market",
    r"See the list now",
    r"See what 10M\+ investors are talking about",
    r"Get the Stocktwits Daily Rip",
    r"Building a portfolio and living off one",
    r"\(Sponsor\)"
]

def build_clean_article_text(row):
    parts = []

    title = row.get("title")
    summary = row.get("summary")
    content = row.get("content")

    if pd.notna(title) and str(title).strip():
        parts.append(f"Title: {str(title).strip()}")

    if pd.notna(summary) and str(summary).strip():
        parts.append(f"Summary: {str(summary).strip()}")

    if pd.notna(content) and str(content).strip():
        paragraphs = re.split(
            r"\n\s*\n",
            str(content).strip()
        )

        clean_paragraphs = [
            paragraph.strip()
            for paragraph in paragraphs
            if paragraph.strip()
            and not any(
                re.search(pattern, paragraph, flags=re.IGNORECASE)
                for pattern in ad_patterns
            )
        ]

        if clean_paragraphs:
            parts.append(
                "Content: " + "\n\n".join(clean_paragraphs)
            )

    return "\n".join(parts)


# Create clean retrieval texts without changing the source DataFrame
news_texts_clean = [
    build_clean_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Retrieval Text Cleaning ==========")
print("Candidate articles:", len(news_texts_clean))
print("Empty article texts:", sum(
    not text.strip() for text in news_texts_clean
))

print("\n========== Cleaned Example ==========")
print(news_texts_clean[2][:1000])

========== Retrieval Text Cleaning ==========
Candidate articles: 124
Empty article texts: 0

========== Cleaned Example ==========
Title: Cleveland Fed President Wants Small, Quick Rate Moves. What It Means for Bitcoin
Summary: Fed’s Hammack backs restrictive policy as December hike odds near 85% and Bitcoin slips in its strongest quarter.
Content: Cleveland Fed President Beth Hammack said that restrictive monetary policy is appropriate to return inflation to 2%. She also favors making smaller rate adjustments quickly rather than waiting and risking a larger correction later.

Rate futures now point to a hike in December, and the US dollar sits near an 18-month high. Meanwhile, Bitcoin (BTC) has slipped in October, historically one of its strongest months.

Hammack, a voting member of the Federal Open Market Committee (FOMC), spoke at the National Association for Business Economics annual meeting. She said the breadth of current price pressures means inflation will take some time to r

In [16]:
# 1. Encode the same forecasting query
query_embedding_clean = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Re-encode the cleaned news texts
news_embeddings_clean = sbert_model.encode(
    news_texts_clean,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Recalculate cosine similarity
clean_scores = util.cos_sim(
    query_embedding_clean,
    news_embeddings_clean
).cpu().numpy().flatten()

# 4. Create a new results DataFrame
cleaned_semantic_results_df = keyword_filtered_df.copy()
cleaned_semantic_results_df["similarity_score"] = clean_scores

cleaned_semantic_results_df = (
    cleaned_semantic_results_df
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

cleaned_semantic_results_df["similarity_rank"] = range(
    1, len(cleaned_semantic_results_df) + 1
)

# 5. Compare old and new rankings
rank_comparison = semantic_results_df[
    ["link", "title", "similarity_rank", "similarity_score"]
].merge(
    cleaned_semantic_results_df[
        ["link", "similarity_rank", "similarity_score"]
    ],
    on="link",
    suffixes=("_before", "_after"),
    validate="one_to_one"
)

# 6. Display the new Top 10
print("========== Cleaned SBERT Retrieval Summary ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("New embeddings shape:", tuple(news_embeddings_clean.shape))
print("Similarity scores recalculated:", len(clean_scores))

print("\n========== Top 10 After Cleaning ==========")
display(
    cleaned_semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "similarity_score"
        ]
    ].head(10)
)

print("\n========== Ranking Comparison ==========")
old_top10_links = set(
    semantic_results_df.head(10)["link"]
)
new_top10_links = set(
    cleaned_semantic_results_df.head(10)["link"]
)

print("Top 10 articles retained:", len(old_top10_links & new_top10_links))
print("Articles entering Top 10:", len(new_top10_links - old_top10_links))
print("Articles leaving Top 10:", len(old_top10_links - new_top10_links))

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.68it/s]

========== Cleaned SBERT Retrieval Summary ==========
Candidate articles: 124
New embeddings shape: (124, 384)
Similarity scores recalculated: 124

========== Top 10 After Cleaning ==========


,similarity_rank,title,source,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,0.626807
1,2,"Bitcoin and ethereum prices today, Monday, Oct...",Yahoo Personal Finance,0.626133
2,3,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,0.613354
3,4,"Bitcoin struggles above $83,000 a year after $...",Investing.com,0.587221
4,5,Bitcoin Ranges Near $83K as Traders Look to US...,decrypt,0.584239
5,6,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,0.576397
6,7,"Why Bitcoin Can Fall Before a Fed Rate Hike, T...",MoneyLion,0.565969
7,8,Bitcoin’s Price Prediction: Will Global M2 Poi...,24/7 Wall St.,0.562907
8,9,"Bitcoin Dips to $80,000: How Other Coins Fared...",24/7 Wall St.,0.556565
9,10,Bitcoin Has Closed Four Straight Weeks Above I...,24/7 Wall St.,0.553092



========== Ranking Comparison ==========
Top 10 articles retained: 7
Articles entering Top 10: 3
Articles leaving Top 10: 3


In [17]:
# Update the final retrieved news using cleaned SBERT results

TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

print("========== Final Retrieved News ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))
print(
    "Similarity scores descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

display(
    retrieved_news_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ]
)

========== Final Retrieved News ==========
Candidate articles: 124
Retrieved articles: 10
Similarity scores descending: True


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,0.626807
1,2,"Bitcoin and ethereum prices today, Monday, Oct...",Yahoo Personal Finance,2026-10-12 11:40:18+00:00,0.626133
2,3,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,0.613354
3,4,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,0.587221
4,5,Bitcoin Ranges Near $83K as Traders Look to US...,decrypt,2026-10-12 11:41:50+00:00,0.584239
5,6,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,0.576397
6,7,"Why Bitcoin Can Fall Before a Fed Rate Hike, T...",MoneyLion,2026-10-12 17:00:27+00:00,0.565969
7,8,Bitcoin’s Price Prediction: Will Global M2 Poi...,24/7 Wall St.,2026-10-11 15:00:21+00:00,0.562907
8,9,"Bitcoin Dips to $80,000: How Other Coins Fared...",24/7 Wall St.,2026-10-10 10:00:32+00:00,0.556565
9,10,Bitcoin Has Closed Four Straight Weeks Above I...,24/7 Wall St.,2026-10-12 12:00:10+00:00,0.553092


In [18]:
print("========== Final SBERT Retrieval Quality Check ==========")

print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))

print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)

print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)

print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)

print(
    "Missing content:",
    retrieved_news_df["content"].isna().sum()
)

print(
    "Missing similarity scores:",
    retrieved_news_df["similarity_score"].isna().sum()
)

print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

print(
    "All scores finite:",
    np.isfinite(retrieved_news_df["similarity_score"]).all()
)

print("\nSimilarity score range:")
print(
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

========== Final SBERT Retrieval Quality Check ==========
Candidate articles: 124
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Missing content: 0
Missing similarity scores: 0
Scores sorted descending: True
All scores finite: True

Similarity score range:
0.5531 to 0.6268


In [19]:
# Map cleaned article texts back to their original article links
clean_text_by_link = dict(
    zip(keyword_filtered_df["link"], news_texts_clean)
)

cleaned_semantic_results_df["retrieval_text"] = (
    cleaned_semantic_results_df["link"].map(clean_text_by_link)
)

# Refresh the final Top 10 using cleaned SBERT rankings
TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

# Select the fields needed for downstream interpretation
columns_to_save = [
    "similarity_rank",
    "similarity_score",
    "title",
    "source",
    "published_at_utc",
    "link",
    "summary",
    "content",
    "retrieval_text",
    "content_status"
]

# Save to the project-level processed news directory
output_dir = Path("../data/news/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bitcoin_news_retrieved_top10.csv"

retrieved_news_df[columns_to_save].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Retrieved news saved successfully.")
print("File:", output_path)
print("Saved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)
print("Columns:", columns_to_save)

Retrieved news saved successfully.
File: ..\data\news\processed\bitcoin_news_retrieved_top10.csv
Saved articles: 10
Duplicate links: 0
Scores sorted descending: True
Columns: ['similarity_rank', 'similarity_score', 'title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'retrieval_text', 'content_status']
